In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

include( "../0-params/rho-list.jl" )

_, base = expfolder( "system" );

In [ ]:
# Hyper parameters.
N = 128;  s = 0.1

# Extract critical noise.
folderlist = [findfolder( N, ρlist[i], s; base=base, crit=true ) for i ∈ 1:Nρ];

In [ ]:
# Import area and perimeter scaling values.
βlist = [vec( readdlm( folder*"area-perimeter-scaling.txt" ) )[1:2]
    for folder ∈ folderlist]

# Unpack the probability map data.
ℓlist = [polyloglaw.( 1:N; β=β ) for β ∈ βlist]
Alist = [(ℓ.^2)./(4π) for ℓ ∈ ℓlist];

In [ ]:
# Import the cluster change frequency values.
θdata = [readdlm( folder*"optimal-rates-hat.txt" ) for folder ∈ folderlist]

# Construct the probability map.
Ωlist = [[P( j; N=N, i=i, λℓ=θlist[i,1], λr=θlist[i,2], κ=θlist[i,3] ) for i in 1:N, j in 1:N]
    for θlist ∈ θdata];

In [ ]:
# Import cluster mass distribution extracted from data.
Flist = [round.( defInt, vec( readdlm( folder*"cluster-frequency.txt" ) ) )
    for folder ∈ folderlist]
flist = [F./sum( F ) for F ∈ Flist];

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ, "Calibrating for each ρ."; dt=1.0 )

# Compute the cluster mass distribution.
T = 1.0;  γ0 = [5.0]
resultslist = Vector{Any}( undef, Nρ )

# Run optimization.
for i ∈ 1:Nρ
    resultslist[i] = optimize( createkernelloss( Flist[i], N, Ωlist[i], Alist[i], ℓlist[i]; T=T, n=64 ), log10.( γ0 ) )
    next!( meter )
end

# Unpack solutions.
γlist = [10.0.^results.minimizer for results ∈ resultslist];

In [ ]:
# Unpack parameters and simulate for plotting.
i = 21;  f = flist[i]
NLL, f̂ = kernelloss( Flist[i], γlist[i], (N, Ωlist[i], Alist[i], ℓlist[i]); T=10T, verbose=true )

# Plot bounds.
xmax = ceil( log10( N ) )
ymin = floor( minimum( log10.( f[f .> 0] ) ) )

# Plot the final distribution from the mean-field equations.
plt = plot(; size=(400,250) )

plot!( plt, [N, N], 10.0.^[ymin,0]; color=:black, lw=2, label="" )

# Plot the CMD from simulations and master equations.
scatter!( plt, f; color=:gray, label="raw data" )
scatter!( plt, f̂; color=:black, lw=2, label="master eq." )

plot!( plt; xlims=10.0.^[0,xmax], xscale=:log10 )
plot!( plt; ylims=10.0.^[ymin,0], yscale=:log10 )

plot!( plt; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)", legend=:bottomleft )

# saveplot( plt, "../figures/cluster-distribution-model_N-$(N).png" )

In [ ]:
# Save the kernel exponents and coefficients.
if true
    for i ∈ 1:Nρ
        writedlm( folderlist[i]*"kernel-parameters-$(length( γlist[i] ))-test.txt", γlist[i] )
    end
end